In [2]:
import duckdb

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA preserve_insertion_order=false")

con.execute("""
COPY (
    SELECT
        t.tx_hash,
        t.block_timestamp,
        t.from_address,
        t.to_address,
        CAST(t.value_wei AS HUGEINT) / 1e18                          AS value_eth,
        (CAST(t.gas_used AS HUGEINT) * CAST(t.gas_price_wei AS HUGEINT)) / 1e18  AS gas_eth,
        t.status,
        t.method_selector
    FROM read_parquet('../data/silver/transactions/**/*.parquet', hive_partitioning=true) t
    INNER JOIN read_parquet('../data/gold/wallet_clusters_analyst.parquet') w
        ON t.from_address = w.address
    WHERE w.is_smart_money = true
)
TO '../data/gold/smart_money_transactions.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

import os
n = duckdb.sql("SELECT COUNT(*) FROM '../data/gold/smart_money_transactions.parquet'").fetchone()[0]
n_wallets = duckdb.sql("SELECT COUNT(DISTINCT from_address) FROM '../data/gold/smart_money_transactions.parquet'").fetchone()[0]
size_mb = os.path.getsize('../data/gold/smart_money_transactions.parquet') / 1024**2
print(f"Smart money transactions: {n:,} rows from {n_wallets:,} wallets ({size_mb:.0f} MB)")

Smart money transactions: 0 rows from 0 wallets (0 MB)


In [6]:
import pandas as pd

df = pd.read_parquet('../data/gold/wallet_clusters_analyst.parquet')

print("Columns + dtypes:")
print(df.dtypes)
print(f"\nShape: {df.shape}")
print("\nFirst row:")
print(df.head(1).to_string(index=False))

Columns + dtypes:
address               str
archetype             str
confidence        float32
tx_count            int32
pnl_eth           float32
roi_proxy         float32
is_smart_money       bool
dtype: object

Shape: (6339025, 7)

First row:
                                   address archetype  confidence  tx_count  pnl_eth  roi_proxy  is_smart_money
0xd966b893be4b20467fafb191f4354dfc45983cdc continuum    0.990081         0 0.340079   0.332711           False


In [7]:
import pandas as pd

df = pd.read_parquet('../data/gold/wallet_clusters_analyst.parquet')

print(f"Total rows: {len(df):,}")
print(f"Unique archetypes: {df['archetype'].nunique()}\n")

print("Archetype distribution:")
counts = df['archetype'].value_counts()
total = len(df)
for arch, n in counts.items():
    pct = 100 * n / total
    bar = '█' * int(40 * n / counts.max())
    print(f"  {arch:<25} {n:>9,} ({pct:5.2f}%)  {bar}")

print(f"\nSmart money wallets: {df['is_smart_money'].sum():,} ({df['is_smart_money'].mean():.2%})")

Total rows: 6,339,025
Unique archetypes: 22

Archetype distribution:
  continuum                 2,421,472 (38.20%)  ████████████████████████████████████████
  mainstream_defi_user        605,387 ( 9.55%)  ██████████
  dormant_retail              567,785 ( 8.96%)  █████████
  long_term_dex_trader        362,831 ( 5.72%)  █████
  whale                       239,792 ( 3.78%)  ███
  receive_only_holder         211,152 ( 3.33%)  ███
  stablecoin_parker           188,981 ( 2.98%)  ███
  power_protocol_user         177,120 ( 2.79%)  ██
  dex_aggregator_user         172,988 ( 2.73%)  ██
  spam_token_sender           167,929 ( 2.65%)  ██
  failed_tx_mev_bot           145,375 ( 2.29%)  ██
  nft_trader                  129,183 ( 2.04%)  ██
  single_purpose_user         123,480 ( 1.95%)  ██
  mev_arb_bot                 110,893 ( 1.75%)  █
  light_eth_user              102,331 ( 1.61%)  █
  spam_sniper_bot             102,229 ( 1.61%)  █
  sybil_wallet                101,360 ( 1.60%)  █
  sporadi

In [8]:
import pandas as pd

df = pd.read_parquet('../data/gold/wallet_clusters_analyst.parquet')

eligible_archetypes = {'whale', 'long_term_dex_trader', 'dex_power_trader', 'power_protocol_user'}

print(f"Total wallets: {len(df):,}\n")

# Filter 1: archetype
mask1 = df['archetype'].isin(eligible_archetypes)
print(f"Pass archetype filter:     {mask1.sum():>10,} ({mask1.mean():.2%})")

# Filter 2: tx_count > 50
mask2 = df['tx_count'] > 50
print(f"Pass tx_count > 50:        {mask2.sum():>10,} ({mask2.mean():.2%})")

# Filter 3: roi_proxy > 0.2
mask3 = df['roi_proxy'] > 0.2
print(f"Pass roi_proxy > 0.2:      {mask3.sum():>10,} ({mask3.mean():.2%})")

# Combined
print(f"\nPass all three:            {(mask1 & mask2 & mask3).sum():>10,}")

# Distribution of roi_proxy for eligible archetypes
print("\nroi_proxy distribution for eligible archetypes:")
print(df.loc[mask1, 'roi_proxy'].describe())

print("\npnl_eth distribution for eligible archetypes:")
print(df.loc[mask1, 'pnl_eth'].describe())

Total wallets: 6,339,025

Pass archetype filter:        864,826 (13.64%)
Pass tx_count > 50:                 0 (0.00%)
Pass roi_proxy > 0.2:         927,807 (14.64%)

Pass all three:                     0

roi_proxy distribution for eligible archetypes:
count    864826.000000
mean         -0.089569
std         731.671326
min     -517192.156250
25%          -0.278666
50%          -0.250177
75%           0.000000
max      299425.125000
Name: roi_proxy, dtype: float64

pnl_eth distribution for eligible archetypes:
count    864826.000000
mean          0.135398
std           0.451338
min          -5.386548
25%           0.119483
50%           0.131132
75%           0.175099
max           4.763433
Name: pnl_eth, dtype: float64


In [10]:
import pandas as pd

clusters = pd.read_parquet('../data/gold/wallet_clusters_full.parquet')
features = pd.read_parquet('../data/gold/wallet_features_v4.parquet')  # the RAW version

print("Sanity check on raw features:")
print(features[['tx_count', 'in_volume_eth', 'total_volume_eth_out']].describe())
# tx_count should range from 10 to millions
# in/out volumes should range from 0 to thousands

ARCHETYPE_MAP = {
    0:  'spam_token_sender', 1: 'spam_sniper_bot', 2: 'meme_trader',
    3:  'failed_tx_mev_bot', 4: 'nft_minter', 5: 'dormant_retail',
    6:  'mev_arb_bot', 7: 'casual_defi_explorer', 8: 'whale',
    9:  'long_term_dex_trader', 10: 'single_purpose_user',
    11: 'mainstream_defi_user', 12: 'dex_power_trader',
    13: 'sybil_wallet', 14: 'receive_only_holder', 15: 'light_eth_user',
    16: 'sporadic_user', 17: 'stablecoin_parker', 18: 'nft_trader',
    19: 'dex_aggregator_user', 20: 'power_protocol_user', -1: 'continuum',
}

SMART_MONEY_ARCHETYPES = {
    'whale', 'long_term_dex_trader', 'dex_power_trader', 'power_protocol_user'
}

signals = clusters.merge(
    features[['address', 'tx_count', 'in_volume_eth', 'total_volume_eth_out']],
    on='address',
    how='left'
)

signals['archetype'] = signals['cluster'].map(ARCHETYPE_MAP).fillna('unknown')
signals['pnl_eth']   = signals['in_volume_eth'] - signals['total_volume_eth_out']
signals['roi_proxy'] = signals['pnl_eth'] / signals['total_volume_eth_out'].replace(0, 1e-9)

signals['is_smart_money'] = (
    signals['archetype'].isin(SMART_MONEY_ARCHETYPES)
    & (signals['tx_count'] > 50)
    & (signals['confidence'] > 0.6)
)

analyst_df = signals[[
    'address', 'archetype', 'confidence', 'tx_count',
    'pnl_eth', 'roi_proxy', 'is_smart_money'
]].copy()

analyst_df['tx_count']       = analyst_df['tx_count'].astype('int32')
analyst_df['pnl_eth']        = analyst_df['pnl_eth'].astype('float32')
analyst_df['roi_proxy']      = analyst_df['roi_proxy'].astype('float32')
analyst_df['confidence']     = analyst_df['confidence'].astype('float32')
analyst_df['is_smart_money'] = analyst_df['is_smart_money'].astype('bool')

analyst_df.to_parquet(
    '../data/gold/wallet_clusters_analyst.parquet',
    compression='zstd', compression_level=9
)

print(f"\nSmart money wallets: {analyst_df['is_smart_money'].sum():,} ({analyst_df['is_smart_money'].mean():.2%})")
print(f"\ntx_count distribution:")
print(analyst_df['tx_count'].describe())
print(f"\npnl_eth distribution:")
print(analyst_df['pnl_eth'].describe())

Sanity check on raw features:
           tx_count  in_volume_eth  total_volume_eth_out
count  6.339025e+06   6.339025e+06          6.339025e+06
mean   8.062152e+01   7.290035e+01          8.874515e+01
std    5.133104e+03   2.735448e+04          2.813740e+04
min    1.000000e+01   0.000000e+00          0.000000e+00
25%    1.300000e+01   0.000000e+00          2.190000e-07
50%    1.900000e+01   4.158055e-03          2.355015e-02
75%    3.700000e+01   1.450226e-01          8.570540e-01
max    4.668572e+06   5.218487e+07          5.189761e+07

Smart money wallets: 139,411 (2.20%)

tx_count distribution:
count    6.339025e+06
mean     8.062152e+01
std      5.133104e+03
min      1.000000e+01
25%      1.300000e+01
50%      1.900000e+01
75%      3.700000e+01
max      4.668572e+06
Name: tx_count, dtype: float64

pnl_eth distribution:
count    6.339025e+06
mean    -1.584480e+01
std      3.971798e+03
min     -6.542429e+06
25%     -9.334669e-02
50%     -1.115000e-06
75%      3.276865e-04
max      4.

In [11]:
import pandas as pd
df = pd.read_parquet('../data/gold/wallet_clusters_analyst.parquet')

print("Extreme outliers (top 10 smart money by absolute |pnl_eth|):")
print(df[df['is_smart_money']]
      .reindex(df[df['is_smart_money']]['pnl_eth'].abs().sort_values(ascending=False).index)
      .head(10)[['address', 'archetype', 'tx_count', 'pnl_eth', 'roi_proxy']]
      .to_string(index=False))

Extreme outliers (top 10 smart money by absolute |pnl_eth|):
                                   address           archetype  tx_count       pnl_eth  roi_proxy
0xeae7380dd4cef6fbd1144f49e4d1e6964258a4f4               whale     88664 -6.542429e+06  -0.352795
0xe69f81b825d7dc31ee9becef4dbeab5cf30e3abb power_protocol_user     32245 -3.907071e+06  -0.978654
0xed0c6079229e2d407672a117c22b62064f4a4312 power_protocol_user      8976 -1.523242e+06  -0.720549
0xa9c61fe59b5702b1d382fd1d5e495887ff34c21d               whale     17118 -1.169098e+06  -0.999267
0xc333e80ef2dec2805f239e3f1e810612d294f771               whale     25363 -9.371088e+05  -0.342423
0xf8191d98ae98d2f7abdfb63a9b0b812b93c873aa               whale    107252 -8.892397e+05  -0.090608
0xb99a2c4c1c4f1fc27150681b740396f6ce1cbcf5 power_protocol_user      6925 -8.437908e+05  -0.699133
0xe287aa11128c7db934722963325146f3efa217b5               whale       608 -7.193324e+05  -0.316041
0xf70da97812cb96acdf810712aa562db8dfa3dbef power_protocol

In [12]:
import pandas as pd

df = pd.read_parquet('../data/gold/wallet_clusters_analyst.parquet')

SMART_MONEY_ARCHETYPES = {
    'whale', 'long_term_dex_trader', 'dex_power_trader', 'power_protocol_user'
}

df['is_smart_money'] = (
    df['archetype'].isin(SMART_MONEY_ARCHETYPES)
    & (df['tx_count'] > 50)
    & (df['confidence'] > 0.6)
    & (df['tx_count'] < 100_000)
    & (df['pnl_eth'].abs() < 50_000)
)

df.to_parquet('../data/gold/wallet_clusters_analyst.parquet',
              compression='zstd', compression_level=9)

print(f"Smart money: {df['is_smart_money'].sum():,} ({df['is_smart_money'].mean():.2%})")
print(f"\nBy archetype:")
print(df[df['is_smart_money']]['archetype'].value_counts())

# Re-check outliers — should be much cleaner now
print(f"\nTop 5 by |pnl_eth| (smart money only):")
sm = df[df['is_smart_money']]
print(sm.reindex(sm['pnl_eth'].abs().sort_values(ascending=False).index)
      .head(5)[['address', 'archetype', 'tx_count', 'pnl_eth']]
      .to_string(index=False))

Smart money: 139,215 (2.20%)

By archetype:
archetype
long_term_dex_trader    45426
whale                   43148
power_protocol_user     40017
dex_power_trader        10624
Name: count, dtype: int64

Top 5 by |pnl_eth| (smart money only):
                                   address           archetype  tx_count       pnl_eth
0xbc8a0f5e41857292d12ef620e2a5895ce65a5ce0               whale     21904 -49409.417969
0xc1c7be6aff14a64396b8e90b424e84a45aaab317 power_protocol_user       544 -48541.351562
0x8e5686ffbab85fe7d181967b1e2ab4ef3491d8b8               whale     14693 -48053.679688
0xaaa47f5d0a4274f20d75820f2b70b4c34086aab1 power_protocol_user       615 -46385.335938
0x28f2e7888bbff2f49e9842e1c6b37fedaff7364c               whale     11874 -46352.406250


In [13]:
import pandas as pd
df = pd.read_parquet('../data/gold/wallet_clusters_analyst.parquet')
print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nColumns: {df.dtypes.to_dict()}")
import os
print(f"\nSize: {os.path.getsize('../data/gold/wallet_clusters_analyst.parquet') / 1024**2:.0f} MB")

                                      address  archetype  confidence  \
0  0xd966b893be4b20467fafb191f4354dfc45983cdc  continuum    0.990081   
1  0xf4d720910ec2580802f8261e59ff7fd9d7252b8b  continuum    0.977997   
2  0x0ebcf4299144991221e82ac7eb922317e585e315  continuum    0.993540   
3  0x254e2535e5464e5ca932c02afc4bd76d683f5006  continuum    0.997765   
4  0xfdba8edf5cf5abc6151fa2a70aa3c2a8a1aa8769  continuum    0.998059   

   tx_count     pnl_eth     roi_proxy  is_smart_money  
0        42   -0.498034 -1.195268e-01           False  
1        39   -2.864854 -9.250178e-01           False  
2      2774 -370.765289 -8.228860e-01           False  
3      2870    3.008458  3.008458e+02           False  
4        83    0.020001  2.000090e+07           False  

Shape: (6339025, 7)

Columns: {'address': <StringDtype(na_value=nan)>, 'archetype': <StringDtype(na_value=nan)>, 'confidence': dtype('float32'), 'tx_count': dtype('int32'), 'pnl_eth': dtype('float32'), 'roi_proxy': dtype('float32'

In [14]:
print(f"\nTop 5 by |pnl_eth| (smart money only):")
sm = df[df['is_smart_money']]
print(sm.reindex(sm['pnl_eth'].abs().sort_values(ascending=False).index)
      .head(5)[['address', 'archetype', 'tx_count', 'pnl_eth']]
      .to_string(index=False))


Top 5 by |pnl_eth| (smart money only):
                                   address           archetype  tx_count       pnl_eth
0xbc8a0f5e41857292d12ef620e2a5895ce65a5ce0               whale     21904 -49409.417969
0xc1c7be6aff14a64396b8e90b424e84a45aaab317 power_protocol_user       544 -48541.351562
0x8e5686ffbab85fe7d181967b1e2ab4ef3491d8b8               whale     14693 -48053.679688
0xaaa47f5d0a4274f20d75820f2b70b4c34086aab1 power_protocol_user       615 -46385.335938
0x28f2e7888bbff2f49e9842e1c6b37fedaff7364c               whale     11874 -46352.406250


In [17]:
import pandas as pd
df = pd.read_parquet('../data/platinum/wallet_clusters_analyst.parquet')
print(f"Total rows: {len(df):,}")
print(f"\nArchetype distribution:")
print(df['archetype'].value_counts(dropna=False))
print(f"\nis_smart_money distribution:")
print(df['is_smart_money'].value_counts(dropna=False))
print(f"\nroi_proxy stats:")
print(df['roi_proxy'].describe())
print(f"\ntx_count stats:")
print(df['tx_count'].describe())

Total rows: 6,339,025

Archetype distribution:
archetype
continuum               2421472
mainstream_defi_user     605387
dormant_retail           567785
long_term_dex_trader     362831
whale                    239792
receive_only_holder      211152
stablecoin_parker        188981
power_protocol_user      177120
dex_aggregator_user      172988
spam_token_sender        167929
failed_tx_mev_bot        145375
nft_trader               129183
single_purpose_user      123480
mev_arb_bot              110893
light_eth_user           102331
spam_sniper_bot          102229
sybil_wallet             101360
sporadic_user            100231
dex_power_trader          85083
nft_minter                83220
meme_trader               73236
casual_defi_explorer      66967
Name: count, dtype: int64

is_smart_money distribution:
is_smart_money
False    6339025
Name: count, dtype: int64

roi_proxy stats:
count    6.339025e+06
mean    -6.094325e-01
std      1.354063e+03
min     -2.672392e+06
25%     -2.781836e-

In [21]:
import pandas as pd

def inspect(path):
    df = pd.read_parquet(path)
    print("Columns:", df.columns.tolist())
    print("\nFirst row:")
    print(df.iloc[0])

In [25]:
import pandas as pd
inspect('../data/platinum/wallet_clusters_analyst.parquet')

Columns: ['address', 'archetype', 'confidence', 'tx_count', 'pnl_eth', 'roi_proxy', 'is_smart_money']

First row:
address           0xd966b893be4b20467fafb191f4354dfc45983cdc
archetype                                          continuum
confidence                                          0.990081
tx_count                                                   0
pnl_eth                                             0.340079
roi_proxy                                           0.332711
is_smart_money                                         False
Name: 0, dtype: object


In [26]:
import os
import pandas as pd

# Check both local M4 and Drive
candidates = [
    '../data/gold/wallet_features.parquet',
    '../data/gold/wallet_features_v4.parquet',
    '../data/gold/wallet_features_raw.parquet',
    '../data/gold/wallet_features_unscaled.parquet',
]

for path in candidates:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024**2
        df = pd.read_parquet(path, columns=['tx_count', 'in_volume_eth', 'total_volume_eth_out']).head(5)
        print(f"\n{path}  ({size:.0f} MB)")
        print(df.to_string(index=False))

ArrowInvalid: No match for FieldRef.Name(in_volume_eth) in address: string
tx_count: int64
account_age_days: int64
__fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string

In [16]:
import duckdb
import os

# ============================================================
# Config
# ============================================================
ANALYST_PARQUET = '../data/platinum/wallet_clusters_analyst.parquet'
SILVER_TRANSACTIONS = '../data/silver/transactions/**/*.parquet'
SILVER_TOKEN_TRANSFERS = '../data/silver/token_transfers/**/*.parquet'

OUT_TX = '../data/platinum/smart_money_transactions.parquet'
OUT_TT = '../data/platinum/smart_money_token_transfers.parquet'

TMP_DIR = '../data/duckdb_tmp'
os.makedirs(TMP_DIR, exist_ok=True)

# ============================================================
# DuckDB setup — memory-safe for M4 Mini
# ============================================================
con = duckdb.connect()
con.execute(f"PRAGMA temp_directory='{TMP_DIR}'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

# ============================================================
# Load smart money wallet set
# ============================================================
print("Loading smart money wallet list...")
con.execute(f"""
    CREATE TEMP TABLE smart_money AS
    SELECT address, archetype, confidence, pnl_eth, roi_proxy
    FROM read_parquet('{ANALYST_PARQUET}')
    WHERE is_smart_money = TRUE
""")
n_smart = con.execute("SELECT COUNT(*) FROM smart_money").fetchone()[0]
print(f"Smart money wallets: {n_smart:,}")

# ============================================================
# Export 1: ETH transactions involving any smart money wallet
# ============================================================
print("\nExporting smart money transactions (ETH txs)...")
con.execute(f"""
COPY (
    SELECT
        t.tx_hash,
        t.block_timestamp,
        t.block_number,
        t.from_address,
        t.to_address,
        CAST(t.value_wei AS HUGEINT) / 1e18           AS value_eth,
        (t.gas_used * CAST(t.effective_gas_price_wei AS HUGEINT)) / 1e18 AS gas_spent_eth,
        t.status,
        t.method_selector,
        sm_from.archetype AS from_archetype,
        sm_to.archetype   AS to_archetype
    FROM read_parquet('{SILVER_TRANSACTIONS}', hive_partitioning=true) t
    LEFT JOIN smart_money sm_from ON t.from_address = sm_from.address
    LEFT JOIN smart_money sm_to   ON t.to_address   = sm_to.address
    WHERE sm_from.address IS NOT NULL OR sm_to.address IS NOT NULL
) TO '{OUT_TX}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

n_tx = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_TX}')").fetchone()[0]
size_mb = os.path.getsize(OUT_TX) / 1024**2
print(f"Saved {n_tx:,} transactions to {OUT_TX}  ({size_mb:.0f} MB)")

# ============================================================
# Export 2: token transfers involving any smart money wallet
# ============================================================
print("\nExporting smart money token transfers...")
con.execute(f"""
COPY (
    SELECT
        tt.transaction_hash AS tx_hash,
        tt.block_timestamp,
        tt.block_number,
        tt.token_address,
        tt.from_address,
        tt.to_address,
        tt.value AS value_raw,
        sm_from.archetype AS from_archetype,
        sm_to.archetype   AS to_archetype
    FROM read_parquet('{SILVER_TOKEN_TRANSFERS}', hive_partitioning=true) tt
    LEFT JOIN smart_money sm_from ON tt.from_address = sm_from.address
    LEFT JOIN smart_money sm_to   ON tt.to_address   = sm_to.address
    WHERE sm_from.address IS NOT NULL OR sm_to.address IS NOT NULL
) TO '{OUT_TT}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

n_tt = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_TT}')").fetchone()[0]
size_mb = os.path.getsize(OUT_TT) / 1024**2
print(f"Saved {n_tt:,} token transfers to {OUT_TT}  ({size_mb:.0f} MB)")

con.close()
print("\nDone.")

Loading smart money wallet list...
Smart money wallets: 0

Exporting smart money transactions (ETH txs)...
Saved 0 transactions to ../data/platinum/smart_money_transactions.parquet  (0 MB)

Exporting smart money token transfers...
Saved 0 token transfers to ../data/platinum/smart_money_token_transfers.parquet  (0 MB)

Done.


In [36]:
full = pd.read_parquet('../data/platinum/wallet_clusters_analyst.parquet').head()

In [37]:
full

,address,archetype,confidence,tx_count,pnl_eth,roi_proxy,is_smart_money
0,0xd966b893be4b20467fafb191f4354dfc45983cdc,continuum,0.990081,0,0.340079,0.332711,False
1,0xf4d720910ec2580802f8261e59ff7fd9d7252b8b,continuum,0.977997,0,-0.954550,-1.185778,False
2,0x0ebcf4299144991221e82ac7eb922317e585e315,continuum,0.993540,5,-0.401976,-0.080395,False
3,0x254e2535e5464e5ca932c02afc4bd76d683f5006,continuum,0.997765,5,1.697902,-3.356265,False
4,0xfdba8edf5cf5abc6151fa2a70aa3c2a8a1aa8769,continuum,0.998059,1,0.151133,-0.293345,False


In [34]:
import pyarrow.parquet as pq
n_rows = pq.read_metadata('../data/platinum/wallet_clusters_full.parquet').num_rows

In [35]:
n_rows

6339025

# Build Smart Money Transactions
* For Time Series

In [41]:
import pandas as pd

df = pd.read_parquet('../data/platinum/wallet_clusters_analyst.parquet')

print(f"Total wallets: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

print(f"\n{'='*60}")
print(f"is_smart_money distribution")
print(f"{'='*60}")
print(df['is_smart_money'].value_counts(dropna=False))
print(f"\nSmart money %: {df['is_smart_money'].mean():.2%}")

print(f"\n{'='*60}")
print(f"Smart money breakdown by archetype")
print(f"{'='*60}")
sm = df[df['is_smart_money']]
print(sm['archetype'].value_counts())
print(f"\nTotal smart money: {len(sm):,}")

print(f"\n{'='*60}")
print(f"Confidence stats — full population vs smart money")
print(f"{'='*60}")
print(pd.DataFrame({
    'all_wallets': df['confidence'].describe(),
    'smart_money': sm['confidence'].describe(),
}).round(3))

print(f"\n{'='*60}")
print(f"Smart money confidence distribution")
print(f"{'='*60}")
print(f"  >0.9 (very high): {(sm['confidence'] > 0.9).sum():,} ({(sm['confidence'] > 0.9).mean():.1%})")
print(f"  0.7-0.9 (high):   {((sm['confidence'] > 0.7) & (sm['confidence'] <= 0.9)).sum():,} "
      f"({((sm['confidence'] > 0.7) & (sm['confidence'] <= 0.9)).mean():.1%})")
print(f"  0.5-0.7 (medium): {((sm['confidence'] > 0.5) & (sm['confidence'] <= 0.7)).sum():,} "
      f"({((sm['confidence'] > 0.5) & (sm['confidence'] <= 0.7)).mean():.1%})")
print(f"  <=0.5 (low):      {(sm['confidence'] <= 0.5).sum():,} "
      f"({(sm['confidence'] <= 0.5).mean():.1%})")

Total wallets: 6,339,025
Columns: ['address', 'archetype', 'confidence', 'is_smart_money']

is_smart_money distribution
is_smart_money
False    5474199
True      864826
Name: count, dtype: int64

Smart money %: 13.64%

Smart money breakdown by archetype
archetype
long_term_dex_trader    362831
whale                   239792
power_protocol_user     177120
dex_power_trader         85083
Name: count, dtype: int64

Total smart money: 864,826

Confidence stats — full population vs smart money
       all_wallets  smart_money
count  6339025.000   864826.000
mean         0.903        0.858
std          0.132        0.136
min          0.295        0.304
25%          0.860        0.780
50%          0.969        0.912
75%          0.996        0.964
max          1.000        1.000

Smart money confidence distribution
  >0.9 (very high): 462,772 (53.5%)
  0.7-0.9 (high):   260,329 (30.1%)
  0.5-0.7 (medium): 139,683 (16.2%)
  <=0.5 (low):      2,042 (0.2%)


In [43]:
import duckdb
import os

# ============================================================
# Config
# ============================================================
ANALYST_PARQUET = '../data/platinum/wallet_clusters_analyst.parquet'
SILVER_TRANSACTIONS = '../data/silver/transactions/**/*.parquet'

OUT_PATH = '../data/platinum/smart_money_transactions.parquet'

TMP_DIR = '../data/duckdb_tmp'
os.makedirs(TMP_DIR, exist_ok=True)

# ============================================================
# DuckDB setup — memory-safe for M4 Mini
# ============================================================
con = duckdb.connect()
con.execute(f"PRAGMA temp_directory='{TMP_DIR}'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

# ============================================================
# Load smart money wallet set (archetype only — no broken numerics)
# ============================================================
print("Loading smart money wallet list...")
con.execute(f"""
    CREATE TEMP TABLE smart_money AS
    SELECT address, archetype
    FROM read_parquet('{ANALYST_PARQUET}')
    WHERE is_smart_money = TRUE
""")
n_smart = con.execute("SELECT COUNT(*) FROM smart_money").fetchone()[0]
print(f"Smart money wallets: {n_smart:,}")

if n_smart == 0:
    raise RuntimeError("No smart money wallets found — check wallet_clusters_analyst.parquet")

# ============================================================
# Export: ETH transactions where smart money is sender OR receiver
# ============================================================
print("\nExporting smart money transactions...")
con.execute(f"""
COPY (
    SELECT
        t.tx_hash,
        t.block_timestamp,
        t.block_number,
        t.from_address,
        t.to_address,
        CAST(t.value_wei AS HUGEINT) / 1e18                                       AS value_eth,
        (CAST(t.gas_used AS HUGEINT) * CAST(t.effective_gas_price_wei AS HUGEINT)) / 1e18  AS gas_spent_eth,
        t.status,
        t.method_selector,
        sm_from.archetype                                                          AS from_archetype,
        sm_to.archetype                                                            AS to_archetype
    FROM read_parquet('{SILVER_TRANSACTIONS}', hive_partitioning=true) t
    LEFT JOIN smart_money sm_from ON t.from_address = sm_from.address
    LEFT JOIN smart_money sm_to   ON t.to_address   = sm_to.address
    WHERE sm_from.address IS NOT NULL OR sm_to.address IS NOT NULL
) TO '{OUT_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

# ============================================================
# Verify
# ============================================================
n_tx = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_PATH}')").fetchone()[0]
size_mb = os.path.getsize(OUT_PATH) / 1024**2

print(f"\nSaved {n_tx:,} transactions to {OUT_PATH}  ({size_mb:.0f} MB)")

# Sanity check
print("\nDirection breakdown:")
con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE from_archetype IS NOT NULL AND to_archetype IS NULL) AS sm_sender_only,
        COUNT(*) FILTER (WHERE from_archetype IS NULL AND to_archetype IS NOT NULL) AS sm_receiver_only,
        COUNT(*) FILTER (WHERE from_archetype IS NOT NULL AND to_archetype IS NOT NULL) AS sm_both_sides
    FROM read_parquet('{OUT_PATH}')
""").fetchdf().pipe(print)

print("\nSample rows:")
con.execute(f"SELECT * FROM read_parquet('{OUT_PATH}') LIMIT 5").fetchdf().pipe(print)

con.close()
print("\nDone.")

Loading smart money wallet list...
Smart money wallets: 864,826

Exporting smart money transactions...

Saved 180,831,786 transactions to ../data/platinum/smart_money_transactions.parquet  (13447 MB)

Direction breakdown:
   sm_sender_only  sm_receiver_only  sm_both_sides
0       113475429          58375998        8980359

Sample rows:
                                             tx_hash     block_timestamp  \
0  0x1f15401cd08be75f76ecb4842760085c6ecca490cdb4... 2025-10-26 09:58:23   
1  0x7ddfe267cfc611052bb1c92b3af3b4a5e3ac7bfea0f4... 2025-10-25 14:10:35   
2  0x8287a396e59b7a5464e6265f25b17b8f3def91620753... 2025-10-15 08:39:23   
3  0x7648d63672c9d218fbf44d2db95e5e2292d194928f0f... 2025-10-26 11:43:35   
4  0x2f74ce2e1eba95dad41cd563f4217ceb8b9f1efce12a... 2025-10-26 13:08:59   

   block_number                                from_address  \
0      23660889  0x396343362be2a4da1ce0c1c210945346fb82aa49   
1      23654981  0x8c8d7c46219d9205f056f28fee5950ad564d7465   
2      23581953 

In [45]:
import duckdb
import os

# ============================================================
# Config
# ============================================================
ANALYST_PARQUET = '../data/platinum/wallet_clusters_analyst.parquet'
SILVER_TRANSACTIONS = '../data/silver/transactions/**/*.parquet'

OUT_PATH = '../data/platinum/smart_money_transactions.parquet'

TMP_DIR = '../data/duckdb_tmp'
os.makedirs(TMP_DIR, exist_ok=True)

# ============================================================
# DuckDB setup — memory-safe for M4 Mini (8GB)
# ============================================================
con = duckdb.connect()
con.execute(f"PRAGMA temp_directory='{TMP_DIR}'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

# ============================================================
# Load smart money set (with confidence)
# ============================================================
print("Loading smart money wallet list...")
con.execute(f"""
    CREATE TEMP TABLE smart_money AS
    SELECT address, archetype, confidence
    FROM read_parquet('{ANALYST_PARQUET}')
    WHERE is_smart_money = TRUE
""")
n_smart = con.execute("SELECT COUNT(*) FROM smart_money").fetchone()[0]
print(f"Smart money wallets: {n_smart:,}")

if n_smart == 0:
    raise RuntimeError("No smart money wallets — check wallet_clusters_analyst.parquet")

# ============================================================
# Export: ETH transactions touching any smart money wallet
# ============================================================
print("\nBuilding smart_money_transactions.parquet ...")
con.execute(f"""
COPY (
    SELECT
        t.tx_hash,
        t.block_timestamp,
        t.block_number,
        t.from_address,
        t.to_address,
        CAST(t.value_wei AS HUGEINT) / 1e18                      AS value_eth,
        (CAST(t.gas_used AS HUGEINT) * CAST(t.effective_gas_price_wei AS HUGEINT)) / 1e18  AS gas_spent_eth,
        t.status,
        t.method_selector,
        sm_from.archetype                                        AS from_archetype,
        sm_to.archetype                                          AS to_archetype,
        sm_from.confidence                                       AS from_confidence,
        sm_to.confidence                                         AS to_confidence
    FROM read_parquet('{SILVER_TRANSACTIONS}', hive_partitioning=true) t
    LEFT JOIN smart_money sm_from ON t.from_address = sm_from.address
    LEFT JOIN smart_money sm_to   ON t.to_address   = sm_to.address
    WHERE sm_from.address IS NOT NULL OR sm_to.address IS NOT NULL
) TO '{OUT_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 22);
""")

# ============================================================
# Verify
# ============================================================
n_tx = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_PATH}')").fetchone()[0]
size_mb = os.path.getsize(OUT_PATH) / 1024**2
print(f"\nSaved {n_tx:,} rows to {OUT_PATH}  ({size_mb:.0f} MB)")

print("\nDirection breakdown:")
print(con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE from_archetype IS NOT NULL AND to_archetype IS NULL)     AS sm_sender_only,
        COUNT(*) FILTER (WHERE from_archetype IS NULL     AND to_archetype IS NOT NULL) AS sm_receiver_only,
        COUNT(*) FILTER (WHERE from_archetype IS NOT NULL AND to_archetype IS NOT NULL) AS sm_both_sides
    FROM read_parquet('{OUT_PATH}')
""").fetchdf().to_string(index=False))

print("\nStatus breakdown (failed-tx check):")
print(con.execute(f"""
    SELECT status, COUNT(*) AS n_txs
    FROM read_parquet('{OUT_PATH}')
    GROUP BY status
    ORDER BY status
""").fetchdf().to_string(index=False))

print("\nSample rows:")
print(con.execute(f"SELECT * FROM read_parquet('{OUT_PATH}') LIMIT 5").fetchdf().to_string(index=False))

con.close()
print("\nDone.")

Loading smart money wallet list...
Smart money wallets: 864,826

Building smart_money_transactions.parquet ...

Saved 180,831,786 rows to ../data/platinum/smart_money_transactions.parquet  (12748 MB)

Direction breakdown:
 sm_sender_only  sm_receiver_only  sm_both_sides
      113475429          58375998        8980359

Status breakdown (failed-tx check):
 status     n_txs
      0   1343376
      1 179488410

Sample rows:
                                                           tx_hash     block_timestamp  block_number                               from_address                                 to_address  value_eth  gas_spent_eth  status method_selector from_archetype to_archetype  from_confidence  to_confidence
0x1f15401cd08be75f76ecb4842760085c6ecca490cdb4b142fe4286387c271837 2025-10-26 09:58:23      23660889 0x396343362be2a4da1ce0c1c210945346fb82aa49 0xb423b53d9076b325c0248d62ef74b11adc211020   0.001507       0.000002       1              0x          whale        whale         0.641

In [46]:
import duckdb
import pandas as pd
import os

# ============================================================
# Config
# ============================================================
ANALYST_PARQUET   = '../data/platinum/wallet_clusters_analyst.parquet'
SM_TX_PARQUET     = '../data/platinum/smart_money_transactions.parquet'
SILVER_TX         = '../data/silver/transactions/**/*.parquet'

con = duckdb.connect()
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

print("=" * 80)
print("SANITY CHECKS — smart_money_transactions.parquet")
print("=" * 80)

# ============================================================
# 1. Row count vs expected order of magnitude
# ============================================================
print("\n[1] Row count and file size")
n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{SM_TX_PARQUET}')").fetchone()[0]
size_mb = os.path.getsize(SM_TX_PARQUET) / 1024**2
print(f"  Rows: {n_rows:,}   File size: {size_mb:.0f} MB")
if n_rows == 0:
    raise RuntimeError("Empty file — investigate before continuing")
if n_rows < 100_000:
    print("  ⚠️  Suspiciously few rows — verify smart-money set is non-empty")

# ============================================================
# 2. Column schema check
# ============================================================
print("\n[2] Schema")
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SM_TX_PARQUET}')").fetchdf()
expected_cols = {
    'tx_hash', 'block_timestamp', 'block_number', 'from_address', 'to_address',
    'value_eth', 'gas_spent_eth', 'status', 'method_selector',
    'from_archetype', 'to_archetype', 'from_confidence', 'to_confidence'
}
actual_cols = set(schema['column_name'])
missing = expected_cols - actual_cols
extra = actual_cols - expected_cols
print(f"  Columns present ({len(actual_cols)}): {sorted(actual_cols)}")
if missing:
    print(f"  ❌ MISSING: {missing}")
if extra:
    print(f"  ℹ️  Extra columns: {extra}")
if not missing and not extra:
    print("  ✅ All expected columns present")

# ============================================================
# 3. Every row has at least one smart-money side
# ============================================================
print("\n[3] Every row must have smart money on at least one side")
n_both_null = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{SM_TX_PARQUET}')
    WHERE from_archetype IS NULL AND to_archetype IS NULL
""").fetchone()[0]
print(f"  Rows with both sides NULL: {n_both_null:,}")
if n_both_null > 0:
    print("  ❌ BUG: rows leaked through that shouldn't have")
else:
    print("  ✅ Filter is correct")

# ============================================================
# 4. Direction breakdown
# ============================================================
print("\n[4] Direction of smart money involvement")
print(con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE from_archetype IS NOT NULL AND to_archetype IS NULL)     AS sm_sender_only,
        COUNT(*) FILTER (WHERE from_archetype IS NULL     AND to_archetype IS NOT NULL) AS sm_receiver_only,
        COUNT(*) FILTER (WHERE from_archetype IS NOT NULL AND to_archetype IS NOT NULL) AS sm_both_sides
    FROM read_parquet('{SM_TX_PARQUET}')
""").fetchdf().to_string(index=False))

# ============================================================
# 5. Unique smart-money wallets actually seen vs in analyst table
# ============================================================
print("\n[5] Smart-money wallet coverage")
sm_total = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{ANALYST_PARQUET}') WHERE is_smart_money = TRUE
""").fetchone()[0]
sm_seen = con.execute(f"""
    SELECT COUNT(DISTINCT addr) FROM (
        SELECT from_address AS addr FROM read_parquet('{SM_TX_PARQUET}') WHERE from_archetype IS NOT NULL
        UNION
        SELECT to_address AS addr FROM read_parquet('{SM_TX_PARQUET}') WHERE to_archetype IS NOT NULL
    )
""").fetchone()[0]
coverage = 100 * sm_seen / sm_total
print(f"  Smart-money wallets in analyst table: {sm_total:,}")
print(f"  Smart-money wallets seen in txs:      {sm_seen:,}  ({coverage:.1f}%)")
if coverage < 50:
    print(f"  ⚠️  Many smart-money wallets have NO transactions — investigate")
elif coverage > 95:
    print(f"  ✅ High coverage")

# ============================================================
# 6. Value/gas decimal scaling sanity
# ============================================================
print("\n[6] Value/gas scaling — should look like reasonable ETH amounts")
print(con.execute(f"""
    SELECT
        ROUND(MIN(value_eth), 6)     AS value_eth_min,
        ROUND(MEDIAN(value_eth), 6)  AS value_eth_median,
        ROUND(MAX(value_eth), 2)     AS value_eth_max,
        ROUND(AVG(value_eth), 4)     AS value_eth_mean,
        ROUND(MIN(gas_spent_eth), 8) AS gas_min,
        ROUND(MEDIAN(gas_spent_eth), 6) AS gas_median,
        ROUND(MAX(gas_spent_eth), 4) AS gas_max
    FROM read_parquet('{SM_TX_PARQUET}')
""").fetchdf().to_string(index=False))
print("  Expected: value_eth median 0.001–10 ish; max could be 1000s")
print("  Expected: gas_median 0.0001–0.01; if >1 ETH something is wrong")

# ============================================================
# 7. Status distribution (failed tx rate)
# ============================================================
print("\n[7] Transaction status distribution")
print(con.execute(f"""
    SELECT status, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{SM_TX_PARQUET}')
    GROUP BY status ORDER BY status
""").fetchdf().to_string(index=False))
print("  Expected: ~85-95% status=1 (success); 5-15% status=0 (failed)")

# ============================================================
# 8. Archetype distribution among smart money
# ============================================================
print("\n[8] Archetype mix (from sender side)")
print(con.execute(f"""
    SELECT from_archetype AS archetype, COUNT(*) AS n_txs,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{SM_TX_PARQUET}')
    WHERE from_archetype IS NOT NULL
    GROUP BY from_archetype
    ORDER BY n_txs DESC
""").fetchdf().to_string(index=False))

# ============================================================
# 9. Date coverage
# ============================================================
print("\n[9] Date range")
print(con.execute(f"""
    SELECT
        MIN(block_timestamp) AS first_ts,
        MAX(block_timestamp) AS last_ts,
        DATE_DIFF('day', MIN(block_timestamp), MAX(block_timestamp)) AS days_covered,
        COUNT(DISTINCT DATE(block_timestamp)) AS unique_days
    FROM read_parquet('{SM_TX_PARQUET}')
""").fetchdf().to_string(index=False))
print("  Expected: ~365 days covered with most days non-empty")

# ============================================================
# 10. Confidence distribution
# ============================================================
print("\n[10] Confidence distribution (from side)")
print(con.execute(f"""
    SELECT
        ROUND(AVG(from_confidence), 3)  AS mean_conf,
        ROUND(MEDIAN(from_confidence), 3) AS median_conf,
        COUNT(*) FILTER (WHERE from_confidence > 0.9) AS n_high_conf,
        COUNT(*) FILTER (WHERE from_confidence < 0.5) AS n_low_conf,
        COUNT(*) AS n_total
    FROM read_parquet('{SM_TX_PARQUET}')
    WHERE from_archetype IS NOT NULL
""").fetchdf().to_string(index=False))

# ============================================================
# 11. Method selector top 10 (interpretability sanity)
# ============================================================
print("\n[11] Top method selectors used by smart money")
print(con.execute(f"""
    SELECT method_selector, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{SM_TX_PARQUET}')
    WHERE from_archetype IS NOT NULL
    GROUP BY method_selector
    ORDER BY n DESC
    LIMIT 10
""").fetchdf().to_string(index=False))
print("  Expected top entries: 0xa9059cbb (transfer), 0x095ea7b3 (approve), 0x3593564c (Uniswap)")

# ============================================================
# 12. Daily flow reconstruction (mirror analyst's query)
# ============================================================
print("\n[12] Sample of analyst's intended daily flow query (last 7 days in data)")
print(con.execute(f"""
    WITH daily AS (
        SELECT
            DATE(block_timestamp) AS date,
            COALESCE(from_archetype, to_archetype) AS archetype,
            SUM(CASE WHEN to_archetype IS NOT NULL THEN value_eth ELSE 0 END)   AS in_eth,
            SUM(CASE WHEN from_archetype IS NOT NULL THEN value_eth + gas_spent_eth ELSE 0 END) AS out_eth,
            COUNT(DISTINCT CASE WHEN to_archetype IS NOT NULL THEN to_address
                                 WHEN from_archetype IS NOT NULL THEN from_address END) AS n_wallets
        FROM read_parquet('{SM_TX_PARQUET}')
        WHERE status = 1
        GROUP BY 1, 2
    )
    SELECT * FROM daily
    WHERE archetype = 'whale'
    ORDER BY date DESC LIMIT 7
""").fetchdf().to_string(index=False))

# ============================================================
# 13. Sample raw rows (eyeball test)
# ============================================================
print("\n[13] First 5 rows (eyeball)")
print(con.execute(f"SELECT * FROM read_parquet('{SM_TX_PARQUET}') LIMIT 5").fetchdf().to_string(index=False))

con.close()
print("\n" + "=" * 80)
print("Sanity checks complete. Review any ⚠️ or ❌ flags above.")
print("=" * 80)

SANITY CHECKS — smart_money_transactions.parquet

[1] Row count and file size
  Rows: 180,831,786   File size: 12748 MB

[2] Schema
  Columns present (13): ['block_number', 'block_timestamp', 'from_address', 'from_archetype', 'from_confidence', 'gas_spent_eth', 'method_selector', 'status', 'to_address', 'to_archetype', 'to_confidence', 'tx_hash', 'value_eth']
  ✅ All expected columns present

[3] Every row must have smart money on at least one side
  Rows with both sides NULL: 0
  ✅ Filter is correct

[4] Direction of smart money involvement
 sm_sender_only  sm_receiver_only  sm_both_sides
      113475429          58375998        8980359

[5] Smart-money wallet coverage
  Smart-money wallets in analyst table: 864,826
  Smart-money wallets seen in txs:      864,826  (100.0%)
  ✅ High coverage

[6] Value/gas scaling — should look like reasonable ETH amounts
 value_eth_min  value_eth_median  value_eth_max  value_eth_mean      gas_min  gas_median  gas_max
           0.0          0.000001  

# REDUCE SMART TRANSCATION SIZE

In [67]:
import duckdb
import os
import time

# ============================================================
# Config
# ============================================================
RAW_PATH       = '../data/platinum/smart_money_transactions.parquet'
OUT_DIR        = '../data/platinum'
TMP_DIR        = '../data/duckdb_tmp'

OUT_DAILY_FLOW    = f'{OUT_DIR}/smart_money_daily_flow.parquet'
OUT_DAILY_METHODS = f'{OUT_DIR}/smart_money_daily_methods.parquet'
OUT_WALLET_DAILY  = f'{OUT_DIR}/smart_money_wallet_daily.parquet'

os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 60)
print("Verifying input file...")
print("=" * 60)
assert os.path.exists(RAW_PATH), f"Not found: {RAW_PATH}"
size_gb = os.path.getsize(RAW_PATH) / 1024**3
print(f"Input file: {RAW_PATH}")
print(f"Size: {size_gb:.1f} GB")

con = duckdb.connect()
con.execute(f"PRAGMA temp_directory='{TMP_DIR}'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RAW_PATH}')").fetchone()[0]
print(f"Total rows: {n_rows:,}")

date_range = con.execute(f"""
    SELECT MIN(block_timestamp), MAX(block_timestamp)
    FROM read_parquet('{RAW_PATH}')
""").fetchone()
print(f"Date range: {date_range[0]} to {date_range[1]}")
print("\nReady to aggregate.\n")

Verifying input file...
Input file: ../data/platinum/smart_money_transactions.parquet
Size: 12.4 GB
Total rows: 180,831,786
Date range: 2025-05-22 00:00:11 to 2026-05-21 23:59:59

Ready to aggregate.



In [68]:
print("=" * 60)
print("Building smart_money_hourly_flow.parquet")
print("=" * 60)
t0 = time.time()

OUT_HOURLY_FLOW = f'{OUT_DIR}/smart_money_hourly_flow.parquet'

con.execute(f"""
COPY (
    SELECT
        DATE_TRUNC('hour', block_timestamp)            AS hour,
        COALESCE(from_archetype, to_archetype)         AS archetype,

        SUM(CASE WHEN to_archetype IS NOT NULL
                 THEN value_eth ELSE 0 END)            AS in_eth,

        SUM(CASE WHEN from_archetype IS NOT NULL
                 THEN value_eth + gas_spent_eth
                 ELSE 0 END)                           AS out_eth,

        SUM(CASE WHEN to_archetype IS NOT NULL
                     THEN value_eth
                 WHEN from_archetype IS NOT NULL
                     THEN -(value_eth + gas_spent_eth)
                 ELSE 0 END)                           AS net_eth,

        COUNT(DISTINCT CASE
                       WHEN to_archetype IS NOT NULL THEN to_address
                       WHEN from_archetype IS NOT NULL THEN from_address
                   END)                                AS n_wallets_active,

        COUNT(*)                                       AS n_txs,
        SUM(CASE WHEN status = 0 THEN 1 ELSE 0 END)    AS n_failed_txs

    FROM read_parquet('{RAW_PATH}')
    WHERE COALESCE(from_archetype, to_archetype) IS NOT NULL
    GROUP BY hour, archetype
    ORDER BY hour, archetype
)
TO '{OUT_HOURLY_FLOW}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

elapsed = time.time() - t0
n_out = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_HOURLY_FLOW}')").fetchone()[0]
size_mb = os.path.getsize(OUT_HOURLY_FLOW) / 1024**2
print(f"Done in {elapsed:.0f}s")
print(f"Output: {OUT_HOURLY_FLOW}")
print(f"Rows: {n_out:,}  Size: {size_mb:.1f} MB")

Building smart_money_hourly_flow.parquet
Done in 39s
Output: ../data/platinum/smart_money_hourly_flow.parquet
Rows: 35,040  Size: 1.0 MB


In [55]:
print("=" * 60)
print("Cell 3: Building smart_money_daily_methods.parquet")
print("=" * 60)
t0 = time.time()

con.execute(f"""
COPY (
    SELECT
        DATE(block_timestamp)                          AS date,
        COALESCE(from_archetype, to_archetype)         AS archetype,
        method_selector,

        SUM(value_eth)                                 AS volume_eth,
        SUM(gas_spent_eth)                             AS gas_eth,
        COUNT(*)                                       AS n_txs,
        COUNT(DISTINCT CASE
                       WHEN to_archetype IS NOT NULL THEN to_address
                       WHEN from_archetype IS NOT NULL THEN from_address
                   END)                                AS n_unique_wallets

    FROM read_parquet('{RAW_PATH}')
    WHERE status = 1
      AND COALESCE(from_archetype, to_archetype) IS NOT NULL
    GROUP BY date, archetype, method_selector
    ORDER BY date, archetype, n_txs DESC
)
TO '{OUT_DAILY_METHODS}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

elapsed = time.time() - t0
n_out = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_DAILY_METHODS}')").fetchone()[0]
size_mb = os.path.getsize(OUT_DAILY_METHODS) / 1024**2
print(f"Done in {elapsed:.0f}s")
print(f"Output: {OUT_DAILY_METHODS}")
print(f"Rows: {n_out:,}  Size: {size_mb:.1f} MB")
print("\nTop method selectors by tx count (overall):")
print(con.execute(f"""
    SELECT method_selector, SUM(n_txs) AS total_txs
    FROM read_parquet('{OUT_DAILY_METHODS}')
    GROUP BY method_selector
    ORDER BY total_txs DESC
    LIMIT 10
""").fetchdf().to_string(index=False))
print()

Cell 3: Building smart_money_daily_methods.parquet
Done in 34s
Output: ../data/platinum/smart_money_daily_methods.parquet
Rows: 3,788,199  Size: 61.2 MB

Top method selectors by tx count (overall):
method_selector   total_txs
             0x 102767879.0
     0xa9059cbb  50981151.0
     0x095ea7b3   3459281.0
     0xdeff4b24    883074.0
     0xb61d27f6    835877.0
     0xa22cb465    488954.0
     0x00000000    444724.0
     0x23b872dd    442224.0
     0x99e1d016    388345.0
     0x34fcd5be    380954.0



In [75]:
import duckdb, os, time

# ============================================================
# Config
# ============================================================
RAW_PATH = '../data/platinum/smart_money_transactions.parquet'
OUT_PATH = '../data/platinum/smart_money_hourly_methods_top3.parquet'

TMP_DIR  = '../data/duckdb_tmp'
os.makedirs(TMP_DIR, exist_ok=True)

TOP_K = 3

print("=" * 60)
print(f"Building smart_money_hourly_methods_top{TOP_K}.parquet")
print("=" * 60)

con = duckdb.connect()
con.execute(f"PRAGMA temp_directory='{TMP_DIR}'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

# ============================================================
# Selector → human signature decoder
# Top 20 by volume in your data; extend as needed.
# ============================================================
print("Loading selector decoder...")
con.execute("""
CREATE TEMP TABLE selector_signatures AS
SELECT * FROM (VALUES
    ('0xa9059cbb', 'transfer(address,uint256)'),
    ('0x095ea7b3', 'approve(address,uint256)'),
    ('0x64617461', 'data_spam'),
    ('0x3593564c', 'execute(bytes,bytes[],uint256)'),
    ('0x23b872dd', 'transferFrom(address,address,uint256)'),
    ('0x2213bc0b', 'unknown_mev_bot_0x2213bc0b'),
    ('0x791ac947', 'swapExactTokensForETHSupportingFeeOnTransferTokens'),
    ('0x78e111f6', 'unknown_sniper_bot_0x78e111f6'),
    ('0x5f575529', 'unknown_router_0x5f575529'),
    ('0x0162e2d0', 'unknown_proxy_0x0162e2d0'),
    ('0x765e827f', 'unknown_0x765e827f'),
    ('0xb6f9de95', 'swapExactETHForTokensSupportingFeeOnTransferTokens'),
    ('0xb143044b', 'unknown_0xb143044b'),
    ('0x00000000', 'empty_or_fallback'),
    ('0xa0000000', 'spam_probe'),
    ('0x1249c58b', 'mint()'),
    ('0x7ff36ab5', 'swapExactETHForTokens(uint256,address[],address,uint256)'),
    ('0x12aa3caf', 'swap_1inch_v4'),
    ('0x07ed2379', 'swap_1inch_aggregator'),
    ('0x771d503f', 'unknown_router_0x771d503f')
) AS t(method_selector, method_signature);
""")
print(f"  {con.execute('SELECT COUNT(*) FROM selector_signatures').fetchone()[0]} selectors decoded\n")

# ============================================================
# Build hourly top-K methods per archetype
# ============================================================
print("Aggregating hourly methods (this is the slow step)...")
t0 = time.time()

con.execute(f"""
COPY (
    WITH hourly_methods AS (
        SELECT
            DATE_TRUNC('hour', block_timestamp)            AS hour,
            COALESCE(from_archetype, to_archetype)         AS archetype,
            method_selector,
            SUM(value_eth)                                 AS volume_eth,
            SUM(gas_spent_eth)                             AS gas_eth,
            COUNT(*)                                       AS n_txs,
            COUNT(DISTINCT CASE
                           WHEN to_archetype IS NOT NULL THEN to_address
                           WHEN from_archetype IS NOT NULL THEN from_address
                       END)                                AS n_unique_wallets
        FROM read_parquet('{RAW_PATH}')
        WHERE status = 1
          AND COALESCE(from_archetype, to_archetype) IS NOT NULL
        GROUP BY hour, archetype, method_selector
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY hour, archetype
                ORDER BY n_txs DESC
            ) AS rank
        FROM hourly_methods
    )
    SELECT
        r.hour,
        r.archetype,
        CASE WHEN r.rank <= {TOP_K} THEN r.method_selector ELSE 'other' END AS method_selector,
        CASE WHEN r.rank <= {TOP_K}
             THEN COALESCE(s.method_signature, 'unknown_' || r.method_selector)
             ELSE 'other' END AS method_signature,
        SUM(r.volume_eth)        AS volume_eth,
        SUM(r.gas_eth)           AS gas_eth,
        SUM(r.n_txs)             AS n_txs,
        SUM(r.n_unique_wallets)  AS n_unique_wallets,    -- approximate (pre-grouped sum)
        MIN(r.rank)              AS rank_in_bucket
    FROM ranked r
    LEFT JOIN selector_signatures s ON r.method_selector = s.method_selector
    GROUP BY r.hour, r.archetype,
             CASE WHEN r.rank <= {TOP_K} THEN r.method_selector ELSE 'other' END,
             CASE WHEN r.rank <= {TOP_K}
                  THEN COALESCE(s.method_signature, 'unknown_' || r.method_selector)
                  ELSE 'other' END
    ORDER BY r.hour, r.archetype, n_txs DESC
)
TO '{OUT_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

elapsed = time.time() - t0
n_out = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_PATH}')").fetchone()[0]
size_mb = os.path.getsize(OUT_PATH) / 1024**2

print(f"Done in {elapsed:.0f}s")
print(f"Output: {OUT_PATH}")
print(f"Rows: {n_out:,}  Size: {size_mb:.2f} MB")

# ============================================================
# Preview
# ============================================================
print("\nPreview — first hour, all archetypes:")
print(con.execute(f"""
    SELECT hour, archetype, method_signature, n_txs, ROUND(volume_eth, 2) AS volume_eth, rank_in_bucket
    FROM read_parquet('{OUT_PATH}')
    WHERE hour = (SELECT MIN(hour) FROM read_parquet('{OUT_PATH}'))
    ORDER BY archetype, rank_in_bucket
""").fetchdf().to_string(index=False))

print("\nSignature coverage (decoded vs unknown):")
print(con.execute(f"""
    SELECT
        CASE WHEN method_signature LIKE 'unknown_%' THEN 'unknown'
             WHEN method_signature = 'other'        THEN 'other'
             ELSE 'decoded' END AS coverage,
        COUNT(*)                AS n_rows,
        SUM(n_txs)              AS total_txs
    FROM read_parquet('{OUT_PATH}')
    GROUP BY coverage
    ORDER BY total_txs DESC
""").fetchdf().to_string(index=False))

con.close()
print("\nDone.")

Building smart_money_hourly_methods_top3.parquet
Loading selector decoder...
  20 selectors decoded

Aggregating hourly methods (this is the slow step)...
Done in 58s
Output: ../data/platinum/smart_money_hourly_methods_top3.parquet
Rows: 140,139  Size: 2.21 MB

Preview — first hour, all archetypes:
      hour            archetype          method_signature  n_txs  volume_eth  rank_in_bucket
2025-05-22     dex_power_trader                unknown_0x   19.0        0.58               1
2025-05-22     dex_power_trader         empty_or_fallback   11.0        1.40               2
2025-05-22     dex_power_trader        unknown_0xa22cb465    9.0        0.00               3
2025-05-22     dex_power_trader                     other   96.0        1.92               4
2025-05-22 long_term_dex_trader transfer(address,uint256)  601.0        0.00               1
2025-05-22 long_term_dex_trader                unknown_0x   76.0        2.93               2
2025-05-22 long_term_dex_trader  approve(address,

In [56]:
print("=" * 60)
print("Cell 4: Building smart_money_wallet_daily.parquet")
print("=" * 60)
t0 = time.time()

con.execute(f"""
COPY (
    SELECT
        DATE(block_timestamp) AS date,
        COALESCE(from_archetype, to_archetype) AS archetype,

        -- Wallet from the smart-money side of the tx
        CASE
            WHEN from_archetype IS NOT NULL THEN from_address
            WHEN to_archetype   IS NOT NULL THEN to_address
        END AS wallet_address,

        -- IN flow (smart money received)
        SUM(CASE WHEN to_archetype IS NOT NULL
                 THEN value_eth ELSE 0 END) AS in_eth,

        -- OUT flow (smart money sent — including gas)
        SUM(CASE WHEN from_archetype IS NOT NULL
                 THEN value_eth + gas_spent_eth
                 ELSE 0 END) AS out_eth,

        COUNT(*) AS n_txs,
        SUM(CASE WHEN status = 0 THEN 1 ELSE 0 END) AS n_failed_txs

    FROM read_parquet('{RAW_PATH}')
    WHERE COALESCE(from_archetype, to_archetype) IS NOT NULL
    GROUP BY date, archetype, wallet_address
    ORDER BY date, archetype, out_eth DESC
)
TO '{OUT_WALLET_DAILY}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

elapsed = time.time() - t0
n_out = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_WALLET_DAILY}')").fetchone()[0]
size_mb = os.path.getsize(OUT_WALLET_DAILY) / 1024**2
print(f"Done in {elapsed:.0f}s")
print(f"Output: {OUT_WALLET_DAILY}")
print(f"Rows: {n_out:,}  Size: {size_mb:.0f} MB")
print("\nUnique wallets seen:")
print(con.execute(f"""
    SELECT archetype, COUNT(DISTINCT wallet_address) AS unique_wallets
    FROM read_parquet('{OUT_WALLET_DAILY}')
    GROUP BY archetype
    ORDER BY unique_wallets DESC
""").fetchdf().to_string(index=False))
print()

Cell 4: Building smart_money_wallet_daily.parquet
Done in 38s
Output: ../data/platinum/smart_money_wallet_daily.parquet
Rows: 18,525,288  Size: 510 MB

Unique wallets seen:
           archetype  unique_wallets
long_term_dex_trader          362831
               whale          239792
 power_protocol_user          177120
    dex_power_trader           85083



In [57]:
print("=" * 60)
print("Summary of shipping bundle")
print("=" * 60)

bundle = [
    (OUT_DAILY_FLOW,    "Daily archetype-level flows (his headline table)"),
    (OUT_DAILY_METHODS, "Daily breakdown by method selector (what they were doing)"),
    (OUT_WALLET_DAILY,  "Per-wallet daily aggregation (for concentration analysis)"),
]

total_mb = 0
for path, desc in bundle:
    size_mb = os.path.getsize(path) / 1024**2
    total_mb += size_mb
    print(f"  {os.path.basename(path):45s} {size_mb:>8.1f} MB  — {desc}")

print(f"\nTotal bundle size: {total_mb:.0f} MB")
print("\nDon't forget to also ship:")
print("  - top_20_selectors.parquet  (decoder for method_selector)")
print("  - cluster_profiles.csv      (archetype definitions)")
print("  - wallet_clusters_analyst.parquet  (the full wallet→archetype table)")
print("  - README.md                 (methodology + caveats)")

con.close()
print("\nDone.")

Summary of shipping bundle
  smart_money_daily_flow.parquet                     0.0 MB  — Daily archetype-level flows (his headline table)
  smart_money_daily_methods.parquet                 61.2 MB  — Daily breakdown by method selector (what they were doing)
  smart_money_wallet_daily.parquet                 510.1 MB  — Per-wallet daily aggregation (for concentration analysis)

Total bundle size: 571 MB

Don't forget to also ship:
  - top_20_selectors.parquet  (decoder for method_selector)
  - cluster_profiles.csv      (archetype definitions)
  - wallet_clusters_analyst.parquet  (the full wallet→archetype table)
  - README.md                 (methodology + caveats)

Done.


In [72]:
# Reducing daily top 20 methods to top k
import duckdb, os, time

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

IN_PATH  = '../data/platinum/smart_money_daily_methods.parquet'
OUT_PATH = '../data/platinum/smart_money_daily_methods_topk.parquet'

TOP_K = 10  # keep top 10 methods per (date, archetype); rest goes to 'other'

print(f"Collapsing to top {TOP_K} methods per (date, archetype)...")
t0 = time.time()

con.execute(f"""
COPY (
    WITH ranked AS (
        SELECT
            date,
            archetype,
            method_selector,
            volume_eth,
            gas_eth,
            n_txs,
            n_unique_wallets,
            ROW_NUMBER() OVER (
                PARTITION BY date, archetype
                ORDER BY n_txs DESC
            ) AS rank
        FROM read_parquet('{IN_PATH}')
    )
    SELECT
        date,
        archetype,
        CASE WHEN rank <= {TOP_K} THEN method_selector ELSE 'other' END AS method_selector,
        SUM(volume_eth)       AS volume_eth,
        SUM(gas_eth)          AS gas_eth,
        SUM(n_txs)            AS n_txs,
        SUM(n_unique_wallets) AS n_unique_wallets,  -- approximate; pre-grouped sums can overcount
        MIN(rank)             AS rank_in_bucket     -- preserves the top rank for the kept rows
    FROM ranked
    GROUP BY date, archetype, CASE WHEN rank <= {TOP_K} THEN method_selector ELSE 'other' END
    ORDER BY date, archetype, n_txs DESC
)
TO '{OUT_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

n_in  = con.execute(f"SELECT COUNT(*) FROM read_parquet('{IN_PATH}')").fetchone()[0]
n_out = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT_PATH}')").fetchone()[0]
size_mb = os.path.getsize(OUT_PATH) / 1024**2

print(f"Done in {time.time()-t0:.0f}s")
print(f"Rows: {n_in:,} → {n_out:,} ({100*(1 - n_out/n_in):.0f}% reduction)")
print(f"Size: {size_mb:.2f} MB")

print("\nSample (one day, all archetypes):")
print(con.execute(f"""
    SELECT date, archetype, method_selector, n_txs, volume_eth, rank_in_bucket
    FROM read_parquet('{OUT_PATH}')
    WHERE date = (SELECT MIN(date) FROM read_parquet('{OUT_PATH}'))
    ORDER BY archetype, n_txs DESC
""").fetchdf().to_string(index=False))

Collapsing to top 10 methods per (date, archetype)...
Done in 0s
Rows: 3,788,199 → 16,060 (100% reduction)
Size: 0.26 MB

Sample (one day, all archetypes):
      date            archetype method_selector    n_txs   volume_eth  rank_in_bucket
2025-05-22     dex_power_trader           other   1664.0 6.738823e+01              11
2025-05-22     dex_power_trader              0x    818.0 9.525966e+01               1
2025-05-22     dex_power_trader      0x00000000    303.0 4.216829e+01               2
2025-05-22     dex_power_trader      0x095ea7b3    289.0 0.000000e+00               3
2025-05-22     dex_power_trader      0xa22cb465    161.0 0.000000e+00               4
2025-05-22     dex_power_trader      0xa9059cbb    143.0 0.000000e+00               5
2025-05-22     dex_power_trader      0x2e1a7d4d    113.0 0.000000e+00               6
2025-05-22     dex_power_trader      0xe7acab24     95.0 2.678759e+00               7
2025-05-22     dex_power_trader      0x0ed64eff     87.0 3.866826e+00 

In [73]:
import duckdb
con = duckdb.connect()
print("Output stats:")
print(con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT date) AS unique_dates,
        COUNT(DISTINCT archetype) AS unique_archetypes,
        COUNT(DISTINCT method_selector) AS unique_methods
    FROM read_parquet('../data/platinum/smart_money_daily_methods_topk.parquet')
""").fetchdf().to_string(index=False))

Output stats:
 total_rows  unique_dates  unique_archetypes  unique_methods
      16060           365                  4             303


In [78]:
pd.read_parquet('../data/platinum/wallet_clusters_analyst.parquet').head()

,address,archetype,confidence,is_smart_money
0,0xd966b893be4b20467fafb191f4354dfc45983cdc,continuum,0.990081,False
1,0xf4d720910ec2580802f8261e59ff7fd9d7252b8b,continuum,0.977997,False
2,0x0ebcf4299144991221e82ac7eb922317e585e315,continuum,0.993540,False
3,0x254e2535e5464e5ca932c02afc4bd76d683f5006,continuum,0.997765,False
4,0xfdba8edf5cf5abc6151fa2a70aa3c2a8a1aa8769,continuum,0.998059,False


In [77]:
pd.read_parquet('../data/platinum/smart_money_daily_flow.parquet').head()

,date,archetype,in_eth,out_eth,net_eth,n_wallets_active,n_txs,n_failed_txs
0,2025-05-22,dex_power_trader,4.320048e+01,1.855108e+02,-138.958097,2030,3937,110.0
1,2025-05-22,long_term_dex_trader,6.982024e+01,6.858542e+01,32.640464,14116,23644,32.0
2,2025-05-22,power_protocol_user,2.190987e+03,3.275593e+04,-29699.657860,8618,45118,2100.0
3,2025-05-22,whale,1.242689e+06,1.402814e+06,752649.111998,14329,297587,1825.0
4,2025-05-23,dex_power_trader,2.978183e+01,2.589078e+02,-224.955588,2109,4567,39.0


In [76]:
pd.read_parquet('../data/platinum/smart_money_hourly_flow.parquet').head()

,hour,archetype,in_eth,out_eth,net_eth,n_wallets_active,n_txs,n_failed_txs
0,2025-05-22 00:00:00,dex_power_trader,0.042044,3.899220,-3.838817,74,135,0.0
1,2025-05-22 00:00:00,long_term_dex_trader,2.221412,1.805914,1.441869,558,679,0.0
2,2025-05-22 00:00:00,power_protocol_user,35.520038,1085.529536,-1025.542181,544,1819,81.0
3,2025-05-22 00:00:00,whale,32454.318498,38375.040714,18437.098237,1715,10254,106.0
4,2025-05-22 01:00:00,dex_power_trader,0.497835,4.545917,-4.034268,107,190,0.0


In [70]:
pd.read_parquet('../data/platinum/smart_money_daily_methods.parquet')

,date,archetype,method_selector,volume_eth,gas_eth,n_txs,n_unique_wallets
0,2025-05-22,dex_power_trader,0x,95.259660,0.075404,818,591
1,2025-05-22,dex_power_trader,0x00000000,42.168285,0.246430,303,212
2,2025-05-22,dex_power_trader,0x095ea7b3,0.000000,0.055258,289,235
3,2025-05-22,dex_power_trader,0xa22cb465,0.000000,0.036945,161,125
4,2025-05-22,dex_power_trader,0xa9059cbb,0.000000,0.026147,143,125
...,...,...,...,...,...,...,...
3788194,2026-05-21,whale,0xc280c905,0.000000,0.000053,1,1
3788195,2026-05-21,whale,0xe046c7b2,0.008354,0.000004,1,1
3788196,2026-05-21,whale,0x1794958f,0.000000,0.000081,1,1
3788197,2026-05-21,whale,0x87201b41,0.191667,0.005054,1,1


In [79]:
pd.read_parquet('../data/platinum/smart_money_hourly_methods_top3.parquet')

,hour,archetype,method_selector,method_signature,volume_eth,gas_eth,n_txs,n_unique_wallets,rank_in_bucket
0,2025-05-22 00:00:00,dex_power_trader,other,other,1.917489,0.019645,96.0,86.0,4
1,2025-05-22 00:00:00,dex_power_trader,0x,unknown_0x,0.581237,0.000547,19.0,14.0,1
2,2025-05-22 00:00:00,dex_power_trader,0x00000000,empty_or_fallback,1.400647,0.002964,11.0,10.0,2
3,2025-05-22 00:00:00,dex_power_trader,0xa22cb465,unknown_0xa22cb465,0.000000,0.000666,9.0,8.0,3
4,2025-05-22 00:00:00,long_term_dex_trader,0xa9059cbb,"transfer(address,uint256)",0.000000,0.068681,601.0,524.0,1
...,...,...,...,...,...,...,...,...,...
140134,2026-05-21 23:00:00,power_protocol_user,0x0a2b8f36,unknown_0x0a2b8f36,0.000939,0.008137,128.0,9.0,3
140135,2026-05-21 23:00:00,whale,0x,unknown_0x,10709.979568,0.134715,18559.0,3317.0,1
140136,2026-05-21 23:00:00,whale,0xa9059cbb,"transfer(address,uint256)",0.000000,0.127214,3606.0,496.0,2
140137,2026-05-21 23:00:00,whale,other,other,62.541928,0.181003,406.0,101.0,4


In [81]:
pd.read_parquet('../data/platinum/wallet_clusters_analyst.parquet').head()

,address,archetype,confidence,is_smart_money
0,0xd966b893be4b20467fafb191f4354dfc45983cdc,continuum,0.990081,False
1,0xf4d720910ec2580802f8261e59ff7fd9d7252b8b,continuum,0.977997,False
2,0x0ebcf4299144991221e82ac7eb922317e585e315,continuum,0.993540,False
3,0x254e2535e5464e5ca932c02afc4bd76d683f5006,continuum,0.997765,False
4,0xfdba8edf5cf5abc6151fa2a70aa3c2a8a1aa8769,continuum,0.998059,False


In [80]:
import pandas as pd
df = pd.read_parquet('../data/platinum/wallet_clusters_analyst.parquet')

print(f"Columns: {df.columns.tolist()}")
print(f"Rows: {len(df):,}")
print(f"\nArchetype distribution:")
print(df['archetype'].value_counts().head(5))
print(f"\nSmart money wallets: {df['is_smart_money'].sum():,}")
print(f"Confidence range: {df['confidence'].min():.3f} to {df['confidence'].max():.3f}")

Columns: ['address', 'archetype', 'confidence', 'is_smart_money']
Rows: 6,339,025

Archetype distribution:
archetype
continuum               2421472
mainstream_defi_user     605387
dormant_retail           567785
long_term_dex_trader     362831
whale                    239792
Name: count, dtype: int64

Smart money wallets: 864,826
Confidence range: 0.295 to 1.000
